This is a file for prototyping using the models/dataloaders defined elsewhere. In addition, it contains the training loop but should NOT include the actual models themselves

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, Dataset
# from torchvision import transforms
# from torchvision.transforms import RandAugment
import transformers

from transformers import get_cosine_schedule_with_warmup


import random

from tqdm import tqdm
import sys


# from google.colab import drive

torch.manual_seed(0)

# drive.mount('/content/drive', force_remount=True)


base_dir = "/content/drive/MyDrive/Junior/Second Semester/CS 4782 - DL/Final Project - HRM"
# sys.path.append(base_dir)
from Model.HRM_Model import HRM
from Model.HRM_Components import Encoder, HighLevel, LowLevel, Head
from Datasets.Sudoku_DataLoader import get_loaders

# sys.path.append(base_dir)
device = torch.device("cuda" if torch.cuda.is_available() else 
                        ("mps" if torch.backends.mps.is_available() else"cpu"))
# device = torch.device("meta")
print(F"Device set to {device}")

Device set to mps


In [10]:
train_size = 1024
test_size = 100
batch_size = 64
train_dataloader, val_dataloader = get_loaders(train_size, test_size, batch_size)


Map: 100%|██████████| 100/100 [00:00<00:00, 10178.13 examples/s]


In [11]:
#Define Model Hyperparameters
d_model = 512
M = 2
N = 2
T = 2
n_layers = 4
n_heads = 8
vocab_size = 10
lr = 1e-4
beta1 = .9
beta2 = .95
weight_decay = .1
num_epochs = 100
num_warmup_steps = 2000

high_level = HighLevel(d_model=d_model, n_layers=n_layers, n_heads=n_heads, intermediate_size=4*d_model)
low_level = HighLevel(d_model=d_model, n_layers=n_layers, n_heads=n_heads, intermediate_size=4*d_model)
head = Head(vocab_size=vocab_size, d_model=d_model)
encoder = Encoder(vocab_size=vocab_size, d_model=d_model, max_len=81)


HRM_model = HRM(low_level, high_level, encoder, head, M, N, T).to(device)

# Define the optimizer
optimizer = optim.AdamW(
    HRM_model.parameters(),
    lr=lr,
    betas=(beta1, beta2),
    weight_decay=weight_decay
)
num_training_steps = len(train_dataloader) * num_epochs
num_warmup_steps = num_warmup_steps

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
    num_cycles=0.5  # standard cosine
)

In [12]:
def train_hrm_deepsup(model, train_loader, optimizer, loss_fn, scheduler=None, num_epochs=10, vocab_size=10):
    """
    Train HRM with deep supervision for multiple epochs and report stats.

    model      : HRM model
    loader     : DataLoader with (input_seq, target_seq)
    optimizer  : optimizer
    loss_fn    : loss function (nn.CrossEntropyLoss)
    scheduler  : optional learning rate scheduler
    num_epochs : number of epochs to train
    vocab_size : size of discrete token vocabulary
    """
    model.train()
    print(f"Number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    for epoch in range(num_epochs):
        correct_tokens = 0
        total_tokens = 0
        epoch_loss = 0

        for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            x = x.to(device)
            y = y.to(device)
            x_embed = model.encode(x)  # trainable encoder
            B, L, d = x_embed.shape
            z_H = torch.zeros(B, L, d, device=x.device)
            z_L = torch.zeros(B, L, d, device=x.device)
            x_flat = x.view(-1)
            y_flat = y.view(-1)
            mask = (x_flat != y_flat)
            targets = y_flat.clone()
            targets[~mask] = -100

            optimizer.zero_grad()
            total_loss_batch = torch.zeros((), device=device)

            for m in range(model.M):
                z_H = z_H.detach()
                z_L = z_L.detach()
                # Detach x_embed after first segment so encoder only trained once
                x_in = x_embed if m == 0 else x_embed.detach()
                z_H, z_L, logits = model.step(z_H, z_L, x_in)

                # Segment-wise forward
                # logits = model.head(z_H)  # (B, L, vocab_size)

                pred = logits.view(B * L, vocab_size)

                loss = loss_fn(pred, targets)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                total_loss_batch += (loss.item() / model.M)

            if scheduler is not None:
                scheduler.step()
            epoch_loss += total_loss_batch
            with torch.no_grad():
                # pred_labels = pred.argmax(dim=1)
                probs = torch.softmax(pred, dim=1)
                dist = torch.distributions.Categorical(probs=probs)
                pred_labels = dist.sample()
                correct_tokens += (pred_labels[mask] == y_flat[mask]).sum()
                total_tokens +=  mask.sum()
        avg_loss = epoch_loss / len(train_loader)
        acc = correct_tokens.item() / total_tokens.item()

       
        print(f"Epoch {epoch+1}: Avg Train Loss = {avg_loss:.4f}, Train Token Accuracy = {acc*100:.2f}%")

    return model
train_hrm_deepsup(HRM_model,train_dataloader, optimizer,nn.CrossEntropyLoss(ignore_index=-100), scheduler, num_epochs=num_epochs,
                vocab_size=vocab_size)
HRM_model.eval()

Number of trainable parameters: 25,270,794


Epoch 1/100: 100%|██████████| 16/16 [00:23<00:00,  1.48s/it]


Epoch 1: Avg Train Loss = 2.4074, Train Token Accuracy = 10.36%


Epoch 2/100: 100%|██████████| 16/16 [00:22<00:00,  1.42s/it]


Epoch 2: Avg Train Loss = 2.2782, Train Token Accuracy = 10.58%


...

Epoch 17: Avg Train Loss = 2.1819, Train Token Accuracy = 11.66%


Epoch 18/100: 100%|██████████| 16/16 [00:22<00:00,  1.39s/it]


Epoch 18: Avg Train Loss = 2.1819, Train Token Accuracy = 11.65%


Epoch 19/100:  38%|███▊      | 6/16 [00:09<00:15,  1.52s/it]


KeyboardInterrupt: 

In [14]:
#Visualize a sudoku
data_iter = iter(val_dataloader)
x_batch, y_batch = next(data_iter)
x = x_batch[0]
y = y_batch[0]
# x,y = train_dataset[random.randint(0,train_size-1)]
x = torch.tensor(x, dtype=torch.long)
print("X: " + str(x))
pred = HRM_model.predict(x).to("cpu")
print("Prediction: " + str(pred))
print("Y: " + str(y))
cor_tok = (pred == y).sum()
print("Prediction Accuracy: " + str(cor_tok / 81))


/var/folders/hg/y35ccm617xld9rm686z5l2q80000gn/T/ipykernel_24312/317451330.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.long)


X: tensor([0, 9, 0, 0, 0, 1, 2, 0, 0, 0, 3, 0, 0, 2, 8, 4, 0, 6, 0, 6, 0, 0, 0, 0,
        0, 8, 0, 0, 7, 0, 0, 0, 0, 1, 4, 0, 0, 2, 0, 0, 5, 0, 7, 0, 0, 0, 0, 3,
        0, 0, 0, 0, 0, 2, 0, 1, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 5, 0, 0, 0,
        0, 8, 7, 2, 0, 6, 0, 0, 4])


ValueError: too many values to unpack (expected 2)

In [ ]:
#Save the model
# torch.save(HRM_model.state_dict(), "hrm_model.pt")